# Experiment 16: Cross-Condition Calibration Transfer for Embedding-Level Drift Detection

## 1. Objective

This experiment investigates whether embedding-level drift detection thresholds calibrated on one data distribution remain reliable when applied to a different target distribution.

The study compares Euclidean distance and Ledoit–Wolf-regularized Mahalanobis distance under isotropic, anisotropic, and correlated synthetic embedding distributions.

## 2. Research Questions

1. How does transferring a frozen calibration threshold across distribution conditions affect the false-alarm rate?
2. Does calibration transfer affect drift detection sensitivity?
3. Do Euclidean and Mahalanobis detectors differ in their robustness to cross-condition calibration transfer?
4. How do the results compare with condition-specific calibration thresholds?

## 3. Experimental Methodology

The experiment uses synthetic embeddings with a dimensionality of 64 and a window size of 20 embeddings. The evaluation is designed around 20 random seeds, with separate reference, calibration, control, and evaluation data.

The principal stages are:

- Generate synthetic embedding distributions with different covariance structures.
- Calibrate detectors on a source condition and freeze the resulting thresholds.
- Apply the frozen thresholds to independent control and drift-evaluation windows from target conditions.
- Compare transferred-threshold performance against a condition-specific calibration benchmark.
- Aggregate false-alarm rates and detection rates across random seeds.
- Conduct statistical comparisons and numerical integrity checks.

## 4. Detectors

**Euclidean detector:** Measures the Euclidean distance between a window's mean embedding and the reference mean.

**Mahalanobis detector:** Measures the distance using a regularized covariance estimate, accounting for variation and correlation among embedding dimensions.

## 5. Evaluation Metrics

- False-alarm rate on independent control windows.
- Drift detection rate on evaluation windows with injected distribution shifts.
- Difference in detection rates between detectors and calibration strategies.
- Across-seed variability and confidence intervals.
- Numerical stability and integrity audit results.

## 6. Reproducibility

The experiment records configuration parameters, software versions, random seeds, calibration thresholds, evaluation outputs, and validation checks. Results are saved under `results/experiment_16/`.

## 7. Scope and Limitations

This experiment uses synthetic embedding distributions. It evaluates statistical properties of the detectors and their calibration transfer; it does not establish real-world biometric recognition accuracy, fairness, or deployment readiness.

All findings will be reported from the actual experiment outputs. No results are assumed in advance.

In [1]:

from pathlib import Path
import json
import platform
import numpy as np
import pandas as pd
import scipy
import sklearn

CONFIG = {
    "experiment_id": 16,
    "title": "Cross-Condition Calibration Transfer for Embedding-Level Drift Detection",
    "random_seed": 202616,
    "n_seeds": 20,
    "embedding_dimension": 64,
    "window_size": 20,
    "reference_samples": 1000,
    "calibration_windows": 200,
    "control_windows": 500,
    "evaluation_windows_per_condition": 500,
    "target_false_alarm_rate": 0.05,
    "mean_shift_strength": 0.5,
    "correlation_strength": 0.6,
    "detectors": ["euclidean", "mahalanobis"],
    "conditions": ["isotropic", "anisotropic", "correlated"],
    "calibration_strategy": "source_condition_threshold_frozen"
}

RESULTS_DIR = Path("/content/results/experiment_16")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

with open(RESULTS_DIR / "config.json", "w") as f:
    json.dump(CONFIG, f, indent=2)

environment = {
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scipy": scipy.__version__,
    "scikit_learn": sklearn.__version__,
    "platform": platform.platform()
}

with open(RESULTS_DIR / "environment.json", "w") as f:
    json.dump(environment, f, indent=2)

print("Experiment 16 configuration created.")
print("Results directory:", RESULTS_DIR)
print("Seeds:", CONFIG["n_seeds"])
print("Embedding dimension:", CONFIG["embedding_dimension"])
print("Conditions:", CONFIG["conditions"])
print("Detectors:", CONFIG["detectors"])
print("Target false-alarm rate:", CONFIG["target_false_alarm_rate"])
print("Environment:", environment)


Experiment 16 configuration created.
Results directory: /content/results/experiment_16
Seeds: 20
Embedding dimension: 64
Conditions: ['isotropic', 'anisotropic', 'correlated']
Detectors: ['euclidean', 'mahalanobis']
Target false-alarm rate: 0.05
Environment: {'python': '3.13.16', 'numpy': '2.1.3', 'pandas': '2.2.3', 'scipy': '1.16.3', 'scikit_learn': '1.6.1', 'platform': 'Linux-6.6.159+-x86_64-with-glibc2.39'}


## Step 2: Synthetic Covariance Construction and Numerical Validation

This stage constructs three synthetic embedding distributions: isotropic, anisotropic, and correlated.

- **Isotropic:** All embedding dimensions have equal variance.
- **Anisotropic:** Embedding dimensions have different variances.
- **Correlated:** Embedding dimensions exhibit structured pairwise correlations.

The covariance matrices are checked for symmetry, finite values, and positive definiteness. These checks help ensure that the distributions are numerically valid for subsequent detector calibration and evaluation.

No detection results are generated in this stage.

In [2]:

import numpy as np
import pandas as pd

D = CONFIG["embedding_dimension"]
rho = CONFIG["correlation_strength"]

# 1. Isotropic covariance
cov_isotropic = np.eye(D)

# 2. Anisotropic covariance: variances range from 0.25 to 4.0
variances = np.geomspace(0.25, 4.0, D)
cov_anisotropic = np.diag(variances)

# 3. Correlated covariance: Toeplitz structure
indices = np.arange(D)
cov_correlated = rho ** np.abs(indices[:, None] - indices[None, :])

COVARIANCE_MATRICES = {
    "isotropic": cov_isotropic,
    "anisotropic": cov_anisotropic,
    "correlated": cov_correlated,
}

diagnostics = []

for condition, cov in COVARIANCE_MATRICES.items():
    eigenvalues = np.linalg.eigvalsh(cov)
    diagnostics.append({
        "condition": condition,
        "dimension": cov.shape[0],
        "all_finite": bool(np.isfinite(cov).all()),
        "symmetric": bool(np.allclose(cov, cov.T)),
        "minimum_eigenvalue": float(eigenvalues.min()),
        "maximum_eigenvalue": float(eigenvalues.max()),
        "positive_definite": bool(eigenvalues.min() > 1e-10),
        "condition_number": float(np.linalg.cond(cov)),
    })

covariance_diagnostics = pd.DataFrame(diagnostics)
display(covariance_diagnostics)

assert covariance_diagnostics["all_finite"].all()
assert covariance_diagnostics["symmetric"].all()
assert covariance_diagnostics["positive_definite"].all()

covariance_diagnostics.to_csv(
    RESULTS_DIR / "covariance_diagnostics.csv",
    index=False
)

print("Covariance validation passed.")
print("Saved:", RESULTS_DIR / "covariance_diagnostics.csv")


,condition,dimension,all_finite,symmetric,minimum_eigenvalue,maximum_eigenvalue,positive_definite,condition_number
0,isotropic,64,True,True,1.00000,1.000000,True,1.000000
1,anisotropic,64,True,True,0.25000,4.000000,True,16.000000
2,correlated,64,True,True,0.25014,3.968234,True,15.864041


Covariance validation passed.
Saved: /content/results/experiment_16/covariance_diagnostics.csv


## Step 3: Reference Dataset Generation

This stage generates independent reference embeddings for each covariance condition and random seed.

Each reference dataset contains 1,000 samples with 64 dimensions and is drawn from a zero-mean multivariate normal distribution using the corresponding covariance matrix.

The reference datasets establish the baseline distribution used by the drift detectors. Separate random seeds produce independent datasets to support multi-seed evaluation.

This stage records dataset diagnostics only; it does not calibrate thresholds or evaluate drift detection performance.

In [3]:

# Experiment 16 — Step 3: Generate reference datasets

REFERENCE_DATASETS = {}
reference_diagnostics = []

n_reference = CONFIG["reference_samples"]
D = CONFIG["embedding_dimension"]

for seed in range(CONFIG["random_seed"], CONFIG["random_seed"] + CONFIG["n_seeds"]):
    for condition, covariance in COVARIANCE_MATRICES.items():
        rng = np.random.default_rng(seed * 100 + CONFIG["conditions"].index(condition))

        X_ref = rng.multivariate_normal(
            mean=np.zeros(D),
            cov=covariance,
            size=n_reference
        )

        REFERENCE_DATASETS[(seed, condition)] = X_ref

        reference_diagnostics.append({
            "seed": seed,
            "condition": condition,
            "n_samples": X_ref.shape[0],
            "n_dimensions": X_ref.shape[1],
            "all_finite": bool(np.isfinite(X_ref).all()),
            "mean_abs": float(np.abs(X_ref.mean(axis=0)).mean()),
            "average_sample_variance": float(X_ref.var(axis=0).mean()),
        })

reference_diagnostics_df = pd.DataFrame(reference_diagnostics)

reference_diagnostics_df.to_csv(
    RESULTS_DIR / "reference_dataset_diagnostics.csv",
    index=False
)

print("Reference datasets generated:", len(REFERENCE_DATASETS))
print("Expected datasets:", CONFIG["n_seeds"] * len(CONFIG["conditions"]))
print("Dataset shape:", (n_reference, D))
print("All values finite:", all(
    np.isfinite(X).all() for X in REFERENCE_DATASETS.values()
))
print("\nDiagnostics by condition:")
display(
    reference_diagnostics_df.groupby("condition")[
        ["n_samples", "mean_abs", "average_sample_variance"]
    ].mean().round(4)
)

assert len(REFERENCE_DATASETS) == CONFIG["n_seeds"] * len(CONFIG["conditions"])
assert all(X.shape == (n_reference, D) for X in REFERENCE_DATASETS.values())
assert all(np.isfinite(X).all() for X in REFERENCE_DATASETS.values())

print("Reference dataset validation passed.")
print("Saved:", RESULTS_DIR / "reference_dataset_diagnostics.csv")


Reference datasets generated: 60
Expected datasets: 60
Dataset shape: (1000, 64)
All values finite: True

Diagnostics by condition:


,n_samples,mean_abs,average_sample_variance
condition,,,
anisotropic,1000.0,0.0275,1.3629
correlated,1000.0,0.0244,0.9971
isotropic,1000.0,0.0246,0.9990


Reference dataset validation passed.
Saved: /content/results/experiment_16/reference_dataset_diagnostics.csv


## Step 4: Cross-Condition Calibration-Transfer Design

This stage defines the source and target distribution pairs used to evaluate calibration transfer.

The three source conditions are isotropic, anisotropic, and correlated. Each detector will be calibrated on a source condition, and its threshold will then be frozen before evaluation on independent target-condition data.

All nine source-target pairs will be evaluated, including the three same-condition pairs as reference cases and the six cross-condition pairs as transfer cases.

For each pair, the experiment will distinguish between:

- Independent target-condition control windows without injected drift.
- Target-condition evaluation windows with controlled mean shifts.

A condition-specific calibration benchmark will also be included to help distinguish threshold-transfer effects from inherent differences between detectors.

The source and target reference datasets remain separate by condition and seed. Thresholds will not be adjusted using target evaluation outcomes.

In [4]:

# Experiment 16 — Step 4: Define source-target transfer scenarios

CONDITIONS = CONFIG["conditions"]

TRANSFER_SCENARIOS = [
    {
        "source_condition": source,
        "target_condition": target,
        "scenario_type": (
            "same_condition"
            if source == target
            else "cross_condition"
        ),
    }
    for source in CONDITIONS
    for target in CONDITIONS
]

transfer_scenarios_df = pd.DataFrame(TRANSFER_SCENARIOS)

transfer_scenarios_df.to_csv(
    RESULTS_DIR / "calibration_transfer_scenarios.csv",
    index=False
)

print("Source-target scenarios:", len(transfer_scenarios_df))
print("Same-condition scenarios:",
      (transfer_scenarios_df["scenario_type"] == "same_condition").sum())
print("Cross-condition scenarios:",
      (transfer_scenarios_df["scenario_type"] == "cross_condition").sum())

display(transfer_scenarios_df)

assert len(transfer_scenarios_df) == 9
assert (transfer_scenarios_df["scenario_type"] == "same_condition").sum() == 3
assert (transfer_scenarios_df["scenario_type"] == "cross_condition").sum() == 6

print("Calibration-transfer scenario validation passed.")
print("Saved:", RESULTS_DIR / "calibration_transfer_scenarios.csv")


Source-target scenarios: 9
Same-condition scenarios: 3
Cross-condition scenarios: 6


,source_condition,target_condition,scenario_type
0,isotropic,isotropic,same_condition
1,isotropic,anisotropic,cross_condition
2,isotropic,correlated,cross_condition
3,anisotropic,isotropic,cross_condition
4,anisotropic,anisotropic,same_condition
5,anisotropic,correlated,cross_condition
6,correlated,isotropic,cross_condition
7,correlated,anisotropic,cross_condition
8,correlated,correlated,same_condition


Calibration-transfer scenario validation passed.
Saved: /content/results/experiment_16/calibration_transfer_scenarios.csv


## Step 5: Detector Scoring and Frozen Threshold Calibration

This stage implements two embedding-level drift detectors: Euclidean distance and Ledoit–Wolf-regularized Mahalanobis distance.

Each incoming window is represented by its mean embedding. The detector calculates a score measuring its deviation from the reference distribution.

For each random seed, source condition, and detector, the calibration scores are calculated using independent windows generated from the source distribution. A threshold is then selected using the empirical 95th percentile, corresponding to the target false-alarm rate of 5%.

The resulting thresholds are frozen before evaluation on target-condition control and drift windows. Target evaluation data will not be used to tune transferred thresholds.

Calibration scores, threshold values, and supporting metadata are saved for reproducibility.

In [5]:

# Experiment 16 — Step 5: Detector scoring and source calibration

from sklearn.covariance import LedoitWolf

def make_window_means(X, window_size, n_windows, rng):
    """Generate independent windows and return each window's mean embedding."""
    windows = rng.multivariate_normal(
        mean=np.zeros(X.shape[1]),
        cov=np.cov(X, rowvar=False),
        size=(n_windows, window_size)
    )
    return windows.mean(axis=1)


def generate_windows(mean, covariance, n_windows, window_size, rng):
    """Generate window-level embeddings from a specified distribution."""
    samples = rng.multivariate_normal(
        mean=mean,
        cov=covariance,
        size=(n_windows, window_size)
    )
    return samples.mean(axis=1)


def fit_detector(reference):
    """Fit reference mean and regularized covariance."""
    reference_mean = reference.mean(axis=0)
    lw = LedoitWolf().fit(reference)
    precision = lw.precision_
    return reference_mean, precision


def detector_scores(window_means, reference_mean, precision, detector):
    """Calculate Euclidean or Mahalanobis scores."""
    delta = window_means - reference_mean

    if detector == "euclidean":
        return np.linalg.norm(delta, axis=1)

    if detector == "mahalanobis":
        squared = np.einsum("ni,ij,nj->n", delta, precision, delta)
        return np.sqrt(np.maximum(squared, 0.0))

    raise ValueError(f"Unknown detector: {detector}")


CALIBRATION_ROWS = []
THRESHOLD_ROWS = []
FITTED_DETECTORS = {}

n_calibration = CONFIG["calibration_windows"]
window_size = CONFIG["window_size"]
target_fpr = CONFIG["target_false_alarm_rate"]

for seed in range(
    CONFIG["random_seed"],
    CONFIG["random_seed"] + CONFIG["n_seeds"]
):
    for source in CONDITIONS:
        reference = REFERENCE_DATASETS[(seed, source)]
        ref_mean, precision = fit_detector(reference)
        FITTED_DETECTORS[(seed, source)] = (ref_mean, precision)

        rng = np.random.default_rng(seed * 10000 + CONDITIONS.index(source))

        calibration_means = generate_windows(
            mean=np.zeros(D),
            covariance=COVARIANCE_MATRICES[source],
            n_windows=n_calibration,
            window_size=window_size,
            rng=rng
        )

        for detector in CONFIG["detectors"]:
            scores = detector_scores(
                calibration_means, ref_mean, precision, detector
            )

            threshold = float(np.quantile(scores, 1 - target_fpr))

            for window_index, score in enumerate(scores):
                CALIBRATION_ROWS.append({
                    "seed": seed,
                    "source_condition": source,
                    "detector": detector,
                    "window_index": window_index,
                    "score": float(score)
                })

            THRESHOLD_ROWS.append({
                "seed": seed,
                "source_condition": source,
                "detector": detector,
                "target_false_alarm_rate": target_fpr,
                "threshold": threshold,
                "calibration_windows": n_calibration
            })

calibration_df = pd.DataFrame(CALIBRATION_ROWS)
thresholds_df = pd.DataFrame(THRESHOLD_ROWS)

calibration_df.to_csv(
    RESULTS_DIR / "calibration_scores.csv", index=False
)
thresholds_df.to_csv(
    RESULTS_DIR / "frozen_thresholds.csv", index=False
)

print("Calibration score rows:", len(calibration_df))
print("Frozen thresholds:", len(thresholds_df))
print("Expected thresholds:", CONFIG["n_seeds"] * len(CONDITIONS) * len(CONFIG["detectors"]))
print("All calibration scores finite:", np.isfinite(calibration_df["score"]).all())
print("All thresholds finite:", np.isfinite(thresholds_df["threshold"]).all())
display(
    thresholds_df.groupby(["source_condition", "detector"])["threshold"]
    .agg(["mean", "std"])
    .round(4)
)

assert len(thresholds_df) == CONFIG["n_seeds"] * len(CONDITIONS) * len(CONFIG["detectors"])
assert np.isfinite(calibration_df["score"]).all()
assert np.isfinite(thresholds_df["threshold"]).all()

print("Detector calibration validation passed.")
print("Saved calibration scores and frozen thresholds.")


Calibration score rows: 24000
Frozen thresholds: 120
Expected thresholds: 120
All calibration scores finite: True
All thresholds finite: True


mean     std
source_condition detector                   
anisotropic      euclidean    2.4881  0.0331
                 mahalanobis  2.0532  0.0184
correlated       euclidean    2.1902  0.0425
                 mahalanobis  2.0688  0.0313
isotropic        euclidean    2.0565  0.0222
                 mahalanobis  2.0580  0.0237

Detector calibration validation passed.
Saved calibration scores and frozen thresholds.


## Step 6: Independent Control Evaluation Under Calibration Transfer

This stage evaluates the frozen source-condition thresholds on independent target-condition control windows without injected drift.

For each random seed, source condition, target condition, and detector, the experiment calculates the proportion of target control windows whose scores exceed the frozen source threshold.

The same-condition scenarios provide a baseline, while cross-condition scenarios reveal whether transferring a threshold changes the false-alarm rate.

The evaluation uses 500 independent control windows per target condition and does not adjust thresholds based on control outcomes. Results are reported as observed false-alarm rates and compared with the nominal target rate of 5%.

Because the control windows are generated synthetically, these findings measure statistical behavior under the specified simulation assumptions rather than real-world biometric performance.

In [7]:

# Experiment 16 — Step 6: Independent control evaluation

CONTROL_ROWS = []
CONTROL_SUMMARY_ROWS = []

n_control = CONFIG["control_windows"]

threshold_lookup = {
    (int(row.seed), row.source_condition, row.detector): float(row.threshold)
    for row in thresholds_df.itertuples(index=False)
}

for seed in range(
    CONFIG["random_seed"],
    CONFIG["random_seed"] + CONFIG["n_seeds"]
):
    for scenario in TRANSFER_SCENARIOS:
        source = scenario["source_condition"]
        target = scenario["target_condition"]

        ref_mean, precision = FITTED_DETECTORS[(seed, source)]

        # Separate reproducible RNG stream for each source-target pair.
        source_idx = CONDITIONS.index(source)
        target_idx = CONDITIONS.index(target)
        rng = np.random.default_rng(
            seed * 100000 + source_idx * 1000 + target_idx + 50000
        )

        control_means = generate_windows(
            mean=np.zeros(D),
            covariance=COVARIANCE_MATRICES[target],
            n_windows=n_control,
            window_size=window_size,
            rng=rng
        )

        for detector in CONFIG["detectors"]:
            threshold = threshold_lookup[(seed, source, detector)]
            scores = detector_scores(
                control_means, ref_mean, precision, detector
            )
            alarms = scores > threshold
            observed_fpr = float(alarms.mean())

            CONTROL_ROWS.append({
                "seed": seed,
                "source_condition": source,
                "target_condition": target,
                "scenario_type": scenario["scenario_type"],
                "detector": detector,
                "window_index": np.arange(n_control),
                "score": scores,
                "threshold": threshold,
                "alarm": alarms
            })

            CONTROL_SUMMARY_ROWS.append({
                "seed": seed,
                "source_condition": source,
                "target_condition": target,
                "scenario_type": scenario["scenario_type"],
                "detector": detector,
                "n_control_windows": n_control,
                "false_alarms": int(alarms.sum()),
                "observed_false_alarm_rate": observed_fpr,
                "target_false_alarm_rate": target_fpr
            })

control_results_df = pd.DataFrame(CONTROL_ROWS)
control_seed_summary_df = pd.DataFrame(CONTROL_SUMMARY_ROWS)

# Expand arrays stored in the rows into one row per window.
control_results_df = control_results_df.explode(
    ["window_index", "score", "alarm"], ignore_index=True
)

control_results_df["window_index"] = control_results_df["window_index"].astype(int)
control_results_df["score"] = control_results_df["score"].astype(float)
control_results_df["threshold"] = control_results_df["threshold"].astype(float)
control_results_df["alarm"] = control_results_df["alarm"].astype(bool)

control_summary_df = (
    control_seed_summary_df
    .groupby(
        ["source_condition", "target_condition", "scenario_type", "detector"],
        as_index=False
    )
    .agg(
        mean_false_alarm_rate=("observed_false_alarm_rate", "mean"),
        sd_false_alarm_rate=("observed_false_alarm_rate", "std"),
        mean_false_alarms=("false_alarms", "mean"),
        total_control_windows=("n_control_windows", "sum")
    )
)

control_results_df.to_csv(
    RESULTS_DIR / "independent_control_results.csv", index=False
)
control_seed_summary_df.to_csv(
    RESULTS_DIR / "independent_control_seed_summary.csv", index=False
)
control_summary_df.to_csv(
    RESULTS_DIR / "independent_control_summary.csv", index=False
)

expected_windows = (
    CONFIG["n_seeds"] * len(TRANSFER_SCENARIOS)
    * len(CONFIG["detectors"]) * n_control
)

print("Independent control windows:", len(control_results_df))
print("Expected control windows:", expected_windows)
print("Seed-level control summaries:", len(control_seed_summary_df))
print("All scores finite:", np.isfinite(control_results_df["score"]).all())
print("All thresholds finite:", np.isfinite(control_results_df["threshold"]).all())
print("\nMean false-alarm rates by transfer scenario:")
display(
    control_summary_df.pivot_table(
        index=["source_condition", "target_condition", "scenario_type"],
        columns="detector",
        values="mean_false_alarm_rate"
    ).round(4)
)

assert len(control_results_df) == expected_windows
assert len(control_seed_summary_df) == (
    CONFIG["n_seeds"] * len(TRANSFER_SCENARIOS) * len(CONFIG["detectors"])
)
assert np.isfinite(control_results_df["score"]).all()
assert np.isfinite(control_results_df["threshold"]).all()

print("Independent control evaluation validation passed.")
print("Saved independent control results and summaries.")


Independent control windows: 180000
Expected control windows: 180000
Seed-level control summaries: 360
All scores finite: True
All thresholds finite: True

Mean false-alarm rates by transfer scenario:


detector                                           euclidean  mahalanobis
source_condition target_condition scenario_type                          
anisotropic      anisotropic      same_condition      0.0507       0.0516
                 correlated       cross_condition     0.0028       0.3538
                 isotropic        cross_condition     0.0002       0.3408
correlated       anisotropic      cross_condition     0.3284       0.9950
                 correlated       same_condition      0.0488       0.0530
                 isotropic        cross_condition     0.0084       0.9667
isotropic        anisotropic      cross_condition     0.5415       0.5410
                 correlated       cross_condition     0.1210       0.1209
                 isotropic        same_condition      0.0575       0.0573

Independent control evaluation validation passed.
Saved independent control results and summaries.


## Step 7: Drift Detection Under Cross-Condition Calibration Transfer

This stage evaluates whether detectors using frozen source-condition thresholds can detect controlled mean shifts in independent target-condition windows.

For each random seed, source condition, target condition, and detector, evaluation windows are generated with a mean shift applied to the target distribution. The source-condition threshold remains frozen throughout evaluation.

The primary metric is the detection rate: the proportion of drift evaluation windows whose detector scores exceed the frozen threshold.

The results will be compared across same-condition and cross-condition scenarios. High detection rates must be interpreted alongside the independent control false-alarm rates, because a detector that alarms on most control windows may also report high detection rates without providing useful discrimination.

This is a synthetic evaluation and does not establish real-world biometric detection performance.

In [8]:

# Experiment 16 — Step 7: Drift evaluation using frozen source thresholds

DRIFT_ROWS = []
DRIFT_SEED_SUMMARY_ROWS = []

n_eval = CONFIG["evaluation_windows_per_condition"]
shift_strength = CONFIG["mean_shift_strength"]

for seed in range(
    CONFIG["random_seed"],
    CONFIG["random_seed"] + CONFIG["n_seeds"]
):
    for scenario in TRANSFER_SCENARIOS:
        source = scenario["source_condition"]
        target = scenario["target_condition"]

        ref_mean, precision = FITTED_DETECTORS[(seed, source)]
        target_cov = COVARIANCE_MATRICES[target]

        # Use a reproducible target-condition drift direction.
        direction_rng = np.random.default_rng(
            seed * 200000
            + CONDITIONS.index(source) * 1000
            + CONDITIONS.index(target)
            + 70000
        )
        direction = direction_rng.normal(size=D)
        direction /= np.linalg.norm(direction)

        shift = shift_strength * direction

        rng = np.random.default_rng(
            seed * 300000
            + CONDITIONS.index(source) * 1000
            + CONDITIONS.index(target)
            + 90000
        )

        drift_means = generate_windows(
            mean=shift,
            covariance=target_cov,
            n_windows=n_eval,
            window_size=window_size,
            rng=rng
        )

        for detector in CONFIG["detectors"]:
            threshold = threshold_lookup[(seed, source, detector)]
            scores = detector_scores(
                drift_means, ref_mean, precision, detector
            )
            alarms = scores > threshold
            detection_rate = float(alarms.mean())

            DRIFT_ROWS.append({
                "seed": seed,
                "source_condition": source,
                "target_condition": target,
                "scenario_type": scenario["scenario_type"],
                "detector": detector,
                "window_index": np.arange(n_eval),
                "score": scores,
                "threshold": threshold,
                "alarm": alarms
            })

            DRIFT_SEED_SUMMARY_ROWS.append({
                "seed": seed,
                "source_condition": source,
                "target_condition": target,
                "scenario_type": scenario["scenario_type"],
                "detector": detector,
                "n_evaluation_windows": n_eval,
                "detected_windows": int(alarms.sum()),
                "detection_rate": detection_rate
            })

drift_results_df = pd.DataFrame(DRIFT_ROWS).explode(
    ["window_index", "score", "alarm"], ignore_index=True
)
drift_results_df["window_index"] = drift_results_df["window_index"].astype(int)
drift_results_df["score"] = drift_results_df["score"].astype(float)
drift_results_df["threshold"] = drift_results_df["threshold"].astype(float)
drift_results_df["alarm"] = drift_results_df["alarm"].astype(bool)

drift_seed_summary_df = pd.DataFrame(DRIFT_SEED_SUMMARY_ROWS)

drift_summary_df = (
    drift_seed_summary_df
    .groupby(
        ["source_condition", "target_condition", "scenario_type", "detector"],
        as_index=False
    )
    .agg(
        mean_detection_rate=("detection_rate", "mean"),
        sd_detection_rate=("detection_rate", "std"),
        mean_detected_windows=("detected_windows", "mean"),
        total_evaluation_windows=("n_evaluation_windows", "sum")
    )
)

drift_results_df.to_csv(
    RESULTS_DIR / "drift_evaluation_results.csv", index=False
)
drift_seed_summary_df.to_csv(
    RESULTS_DIR / "drift_evaluation_seed_summary.csv", index=False
)
drift_summary_df.to_csv(
    RESULTS_DIR / "drift_evaluation_summary.csv", index=False
)

expected_eval_windows = (
    CONFIG["n_seeds"] * len(TRANSFER_SCENARIOS)
    * len(CONFIG["detectors"]) * n_eval
)

print("Drift evaluation windows:", len(drift_results_df))
print("Expected evaluation windows:", expected_eval_windows)
print("Seed-level drift summaries:", len(drift_seed_summary_df))
print("All scores finite:", np.isfinite(drift_results_df["score"]).all())
print("\nMean detection rates by transfer scenario:")
display(
    drift_summary_df.pivot_table(
        index=["source_condition", "target_condition", "scenario_type"],
        columns="detector",
        values="mean_detection_rate"
    ).round(4)
)

assert len(drift_results_df) == expected_eval_windows
assert len(drift_seed_summary_df) == (
    CONFIG["n_seeds"] * len(TRANSFER_SCENARIOS) * len(CONFIG["detectors"])
)
assert np.isfinite(drift_results_df["score"]).all()
assert np.isfinite(drift_results_df["threshold"]).all()

print("Drift evaluation validation passed.")
print("Saved drift evaluation results and summaries.")


Drift evaluation windows: 180000
Expected evaluation windows: 180000
Seed-level drift summaries: 360
All scores finite: True

Mean detection rates by transfer scenario:


detector                                           euclidean  mahalanobis
source_condition target_condition scenario_type                          
anisotropic      anisotropic      same_condition      0.0877       0.1381
                 correlated       cross_condition     0.0060       0.4504
                 isotropic        cross_condition     0.0002       0.4846
correlated       anisotropic      cross_condition     0.4176       0.9984
                 correlated       same_condition      0.0745       0.2194
                 isotropic        cross_condition     0.0295       0.9846
isotropic        anisotropic      cross_condition     0.6427       0.6425
                 correlated       cross_condition     0.1918       0.1917
                 isotropic        same_condition      0.1254       0.1250

Drift evaluation validation passed.
Saved drift evaluation results and summaries.


## Step 8: Statistical Analysis of Calibration Transfer

This stage compares the Euclidean and Mahalanobis detectors across random seeds and source-target calibration scenarios.

The analysis calculates the difference in detection rates between the two detectors for each seed and scenario, using the paired difference:

Mahalanobis detection rate − Euclidean detection rate.

Mean paired differences and 95% confidence intervals are calculated across the 20 seeds. The analysis also reports the corresponding false-alarm rates so that any increase in detection sensitivity can be interpreted alongside the risk of false alarms.

Confidence intervals are unadjusted for multiple comparisons and should be interpreted accordingly. The results measure performance under the synthetic data-generating assumptions used in this experiment.

In [9]:

# Experiment 16 — Step 8: Paired statistical comparison

from scipy import stats

# Paired detection-rate differences for each seed and scenario
detection_pivot = drift_seed_summary_df.pivot(
    index=["seed", "source_condition", "target_condition", "scenario_type"],
    columns="detector",
    values="detection_rate"
).reset_index()

detection_pivot["mahalanobis_minus_euclidean"] = (
    detection_pivot["mahalanobis"] - detection_pivot["euclidean"]
)

difference_rows = []

group_cols = [
    "source_condition",
    "target_condition",
    "scenario_type"
]

for keys, group in detection_pivot.groupby(group_cols):
    diffs = group["mahalanobis_minus_euclidean"].to_numpy()
    n = len(diffs)
    mean_diff = float(np.mean(diffs))
    sd_diff = float(np.std(diffs, ddof=1))
    se_diff = sd_diff / np.sqrt(n)
    critical_t = stats.t.ppf(0.975, df=n - 1)
    ci_low = mean_diff - critical_t * se_diff
    ci_high = mean_diff + critical_t * se_diff

    difference_rows.append({
        "source_condition": keys[0],
        "target_condition": keys[1],
        "scenario_type": keys[2],
        "n_seeds": n,
        "mean_detection_rate_difference": mean_diff,
        "sd_difference": sd_diff,
        "ci95_lower": float(ci_low),
        "ci95_upper": float(ci_high),
        "seeds_mahalanobis_higher": int((diffs > 0).sum()),
        "seeds_euclidean_higher": int((diffs < 0).sum()),
        "seeds_tied": int((diffs == 0).sum())
    })

detection_difference_df = pd.DataFrame(difference_rows)

# Summarize control FPR for both detectors by scenario
fpr_pivot = control_seed_summary_df.pivot(
    index=["seed", "source_condition", "target_condition", "scenario_type"],
    columns="detector",
    values="observed_false_alarm_rate"
).reset_index()

fpr_pivot["mahalanobis_minus_euclidean_fpr"] = (
    fpr_pivot["mahalanobis"] - fpr_pivot["euclidean"]
)

fpr_summary_df = (
    fpr_pivot.groupby(group_cols, as_index=False)
    .agg(
        mean_euclidean_fpr=("euclidean", "mean"),
        mean_mahalanobis_fpr=("mahalanobis", "mean"),
        mean_fpr_difference=("mahalanobis_minus_euclidean_fpr", "mean")
    )
)

detection_difference_df.to_csv(
    RESULTS_DIR / "detection_rate_differences.csv", index=False
)
fpr_summary_df.to_csv(
    RESULTS_DIR / "false_alarm_rate_comparison.csv", index=False
)

print("Paired scenario comparisons:", len(detection_difference_df))
print("Expected comparisons:", len(TRANSFER_SCENARIOS))
print("\nDetection-rate differences and 95% confidence intervals:")
display(
    detection_difference_df[
        group_cols + [
            "mean_detection_rate_difference",
            "ci95_lower",
            "ci95_upper",
            "seeds_mahalanobis_higher",
            "seeds_euclidean_higher"
        ]
    ].round(4)
)

print("\nFalse-alarm rate comparison:")
display(
    fpr_summary_df.round(4)
)

assert len(detection_difference_df) == len(TRANSFER_SCENARIOS)
assert len(fpr_summary_df) == len(TRANSFER_SCENARIOS)
assert np.isfinite(
    detection_difference_df[
        ["mean_detection_rate_difference", "ci95_lower", "ci95_upper"]
    ].to_numpy()
).all()
assert (detection_difference_df["n_seeds"] == CONFIG["n_seeds"]).all()

print("Statistical comparison validation passed.")
print("Saved paired detection differences and false-alarm comparison.")


Paired scenario comparisons: 9
Expected comparisons: 9

Detection-rate differences and 95% confidence intervals:


,source_condition,target_condition,scenario_type,mean_detection_rate_difference,ci95_lower,ci95_upper,seeds_mahalanobis_higher,seeds_euclidean_higher
0,anisotropic,anisotropic,same_condition,0.0504,0.0333,0.0675,19,1
1,anisotropic,correlated,cross_condition,0.4444,0.4256,0.4632,20,0
2,anisotropic,isotropic,cross_condition,0.4844,0.4585,0.5103,20,0
3,correlated,anisotropic,cross_condition,0.5808,0.5462,0.6154,20,0
4,correlated,correlated,same_condition,0.1449,0.1211,0.1687,20,0
5,correlated,isotropic,cross_condition,0.9551,0.9477,0.9625,20,0
6,isotropic,anisotropic,cross_condition,-0.0002,-0.0011,0.0007,2,4
7,isotropic,correlated,cross_condition,-0.0001,-0.0007,0.0005,2,2
8,isotropic,isotropic,same_condition,-0.0004,-0.0009,0.0001,0,3



False-alarm rate comparison:


,source_condition,target_condition,scenario_type,mean_euclidean_fpr,mean_mahalanobis_fpr,mean_fpr_difference
0,anisotropic,anisotropic,same_condition,0.0507,0.0516,0.0009
1,anisotropic,correlated,cross_condition,0.0028,0.3538,0.3510
2,anisotropic,isotropic,cross_condition,0.0002,0.3408,0.3406
3,correlated,anisotropic,cross_condition,0.3284,0.9950,0.6666
4,correlated,correlated,same_condition,0.0488,0.0530,0.0042
5,correlated,isotropic,cross_condition,0.0084,0.9667,0.9583
6,isotropic,anisotropic,cross_condition,0.5415,0.5410,-0.0005
7,isotropic,correlated,cross_condition,0.1210,0.1209,-0.0001
8,isotropic,isotropic,same_condition,0.0575,0.0573,-0.0002


Statistical comparison validation passed.
Saved paired detection differences and false-alarm comparison.


## Step 9: Experiment Integrity Audit

This stage verifies the completeness and numerical validity of the Experiment 16 outputs.

The audit checks the expected counts of reference datasets, calibration scores, frozen thresholds, independent control windows, drift evaluation windows, seed-level summaries, and statistical comparisons.

It also checks for non-finite scores and thresholds, missing values in key result files, and consistency between the configured number of seeds and the generated results.

The audit is intended to identify implementation and record-keeping errors. Passing the audit does not independently establish scientific validity or real-world generalisability.

In [10]:

# Experiment 16 — Step 9: Final integrity audit

audit_checks = []

def record_check(name, passed, observed, expected):
    audit_checks.append({
        "check": name,
        "passed": bool(passed),
        "observed": str(observed),
        "expected": str(expected)
    })

expected_seed_condition_pairs = CONFIG["n_seeds"] * len(CONDITIONS)
expected_thresholds = expected_seed_condition_pairs * len(CONFIG["detectors"])
expected_calibration_scores = expected_thresholds * CONFIG["calibration_windows"]
expected_control_windows = (
    CONFIG["n_seeds"] * len(TRANSFER_SCENARIOS)
    * len(CONFIG["detectors"]) * CONFIG["control_windows"]
)
expected_drift_windows = (
    CONFIG["n_seeds"] * len(TRANSFER_SCENARIOS)
    * len(CONFIG["detectors"])
    * CONFIG["evaluation_windows_per_condition"]
)
expected_summaries = (
    CONFIG["n_seeds"] * len(TRANSFER_SCENARIOS) * len(CONFIG["detectors"])
)

record_check(
    "reference_dataset_count",
    len(REFERENCE_DATASETS) == expected_seed_condition_pairs,
    len(REFERENCE_DATASETS),
    expected_seed_condition_pairs
)
record_check(
    "calibration_score_count",
    len(calibration_df) == expected_calibration_scores,
    len(calibration_df),
    expected_calibration_scores
)
record_check(
    "threshold_count",
    len(thresholds_df) == expected_thresholds,
    len(thresholds_df),
    expected_thresholds
)
record_check(
    "control_window_count",
    len(control_results_df) == expected_control_windows,
    len(control_results_df),
    expected_control_windows
)
record_check(
    "control_summary_count",
    len(control_seed_summary_df) == expected_summaries,
    len(control_seed_summary_df),
    expected_summaries
)
record_check(
    "drift_window_count",
    len(drift_results_df) == expected_drift_windows,
    len(drift_results_df),
    expected_drift_windows
)
record_check(
    "drift_summary_count",
    len(drift_seed_summary_df) == expected_summaries,
    len(drift_seed_summary_df),
    expected_summaries
)
record_check(
    "scenario_count",
    len(transfer_scenarios_df) == 9,
    len(transfer_scenarios_df),
    9
)
record_check(
    "statistical_comparison_count",
    len(detection_difference_df) == 9,
    len(detection_difference_df),
    9
)
record_check(
    "all_calibration_scores_finite",
    np.isfinite(calibration_df["score"]).all(),
    np.isfinite(calibration_df["score"]).all(),
    True
)
record_check(
    "all_thresholds_finite",
    np.isfinite(thresholds_df["threshold"]).all(),
    np.isfinite(thresholds_df["threshold"]).all(),
    True
)
record_check(
    "all_control_scores_finite",
    np.isfinite(control_results_df["score"]).all(),
    np.isfinite(control_results_df["score"]).all(),
    True
)
record_check(
    "all_drift_scores_finite",
    np.isfinite(drift_results_df["score"]).all(),
    np.isfinite(drift_results_df["score"]).all(),
    True
)
record_check(
    "control_rates_within_bounds",
    control_seed_summary_df["observed_false_alarm_rate"].between(0, 1).all(),
    control_seed_summary_df["observed_false_alarm_rate"].between(0, 1).all(),
    True
)
record_check(
    "detection_rates_within_bounds",
    drift_seed_summary_df["detection_rate"].between(0, 1).all(),
    drift_seed_summary_df["detection_rate"].between(0, 1).all(),
    True
)

audit_df = pd.DataFrame(audit_checks)
audit_df.to_csv(
    RESULTS_DIR / "experiment_16_integrity_audit.csv",
    index=False
)

display(audit_df)

print("Checks passed:", int(audit_df["passed"].sum()), "/", len(audit_df))
print("Audit file:", RESULTS_DIR / "experiment_16_integrity_audit.csv")

assert audit_df["passed"].all(), (
    "Integrity audit failed. Inspect the failed checks before proceeding."
)

print("Experiment 16 integrity audit passed.")


,check,passed,observed,expected
0,reference_dataset_count,True,60,60
1,calibration_score_count,True,24000,24000
2,threshold_count,True,120,120
3,control_window_count,True,180000,180000
4,control_summary_count,True,360,360
5,drift_window_count,True,180000,180000
6,drift_summary_count,True,360,360
7,scenario_count,True,9,9
8,statistical_comparison_count,True,9,9
9,all_calibration_scores_finite,True,True,True


Checks passed: 15 / 15
Audit file: /content/results/experiment_16/experiment_16_integrity_audit.csv
Experiment 16 integrity audit passed.


## Step 10: Results Summary and Limitations

### Experiment Overview

Experiment 16 evaluates cross-condition calibration transfer for Euclidean and Ledoit–Wolf-regularized Mahalanobis embedding-level drift detectors using synthetic data.

The evaluation uses 20 random seeds, 64-dimensional embeddings, a window size of 20, and three covariance conditions: isotropic, anisotropic, and correlated. All nine source-target condition pairs are evaluated.

### Experimental Outputs

The experiment generates:
- Reference dataset diagnostics and covariance diagnostics.
- Calibration scores and frozen source-condition thresholds.
- Independent control-window scores and false-alarm summaries.
- Drift evaluation scores and detection-rate summaries.
- Paired detector comparisons with 95% confidence intervals.
- A final integrity audit.

### Preliminary Findings

Same-condition calibration generally produces false-alarm rates near the nominal 5% target. Cross-condition calibration can produce substantial false-alarm inflation or suppression, depending on the source condition, target condition, and detector.

In some transfer scenarios, Mahalanobis exhibits substantially higher detection rates than Euclidean. These increases must be interpreted alongside the corresponding false-alarm rates and do not necessarily indicate better drift discrimination.

### Limitations

The experiment uses synthetic multivariate Gaussian embeddings and controlled mean shifts. It does not evaluate real biometric embeddings, recognition accuracy, fairness, or deployment readiness. The confidence intervals are unadjusted for multiple comparisons.

### Integrity

All 15 programmed integrity checks passed. This establishes that the expected outputs and numerical checks were satisfied; it does not independently establish scientific validity or real-world generalisability.

In [11]:

from pathlib import Path
import zipfile
from google.colab import files

RESULTS_DIR = Path("/content/results/experiment_16")
ZIP_PATH = Path("/content/experiment_16_results.zip")

if not RESULTS_DIR.exists():
    raise FileNotFoundError(f"Results folder not found: {RESULTS_DIR}")

result_files = sorted(
    path for path in RESULTS_DIR.rglob("*") if path.is_file()
)

if not result_files:
    raise RuntimeError("No result files found to package.")

with zipfile.ZipFile(
    ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED
) as zip_file:
    for path in result_files:
        zip_file.write(
            path,
            arcname=str(
                Path("experiment_16") / path.relative_to(RESULTS_DIR)
            )
        )

print("Files packaged:", len(result_files))
print("ZIP size (MB):", round(ZIP_PATH.stat().st_size / (1024 ** 2), 2))
print("ZIP created:", ZIP_PATH)

files.download(str(ZIP_PATH))


Files packaged: 16
ZIP size (MB): 4.85
ZIP created: /content/experiment_16_results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>